# 10 - Linear ΔW coefficient sweep

In [ ]:
import exp_common as E
import pandas as pd
import random, json
from collections import defaultdict, Counter
from pathlib import Path

SWEEP_SEEDS = [42, 123, 777]
LAMBDAS = [0.25, 0.33, 0.5, 0.75, 1.0]
EVAL_LOGS_N = 1000
BATCH_LOGS = 64
EVAL_SEED = 42
USE_CACHE = True

out_dir = E.ANALYSIS_DIR / "coefficient_sweep"
out_dir.mkdir(parents=True, exist_ok=True)

def stratified_logs_sample(path, n_total, seed=42):
    rows = E.read_jsonl(path)
    rng = random.Random(seed)
    buckets = defaultdict(list)
    for r in rows:
        buckets[r['output']].append(r)
    n_per_class = n_total // 2
    out = []
    for label, items in buckets.items():
        k = min(n_per_class, len(items))
        out.extend(rng.sample(items, k))
    rng.shuffle(out)
    return out

logs_eval = stratified_logs_sample(E.PREPARED_DIR / "logs_test_eval.jsonl", EVAL_LOGS_N, seed=EVAL_SEED)
print("Logs eval balance:", dict(Counter(r['output'] for r in logs_eval)))
print(f"Logs eval size: {len(logs_eval)}")

sweep_rows = []

for seed in SWEEP_SEEDS:
    cpt = E.ADAPTERS_DIR / "cpt" / f"seed_{seed}" / "adapter"
    logs = E.ADAPTERS_DIR / "logs_on_cpt" / f"seed_{seed}" / "adapter"

    if not all((p / "adapter_model.safetensors").exists() for p in [cpt, logs]):
        print(f"Skip seed={seed}: missing adapters ({cpt} or {logs})")
        continue

    print(f"\n--- Seed {seed} ---")

    for lam in LAMBDAS:
        adapter_out = E.MERGED_DIR / "coefficient_sweep" / "delta_w" / f"lambda_{lam}" / f"seed_{seed}"
        logs_json = out_dir / f"coefficient_lambda_{lam}_seed_{seed}.json"

        if not (adapter_out / "adapter_model.safetensors").exists():
            print(f"  Merging: lambda={lam}, seed={seed}")
            E.merge_adapters(
                [logs], adapter_out,
                method="task_arithmetic", alpha=lam,
                seed=seed, space="delta_w", target_rank=16
            )
        else:
            print(f"  Merge cached: lambda={lam}, seed={seed}")

        if USE_CACHE and logs_json.exists():
            print(f"  Eval cached: lambda={lam}, seed={seed}")
            m = json.loads(logs_json.read_text(encoding="utf-8"))["metrics"]
        else:
            print(f"  Evaluating: lambda={lam}, seed={seed}")
            m = E.evaluate_logs(
                adapter_out, logs_eval, logs_json,
                batch_size=BATCH_LOGS, base_adapter_dir=cpt
            )

        sweep_rows.append({
            "seed": seed, "lambda": lam,
            **{k: v for k, v in m.items() if not isinstance(v, (list, dict))}
        })

sweep_df = pd.DataFrame(sweep_rows)

if sweep_df.empty:
    raise RuntimeError("No sweep results. Check that CPT and logs_on_cpt adapters exist.")

for seed in SWEEP_SEEDS:
    sub = sweep_df[sweep_df["seed"] == seed]
    if not sub.empty:
        per_seed_path = out_dir / f"coefficient_sweep_seed{seed}.csv"
        sub.to_csv(per_seed_path, index=False)
        print(f"Saved: {per_seed_path.name} ({len(sub)} rows)")

seed_files = sorted(out_dir.glob("coefficient_sweep_seed*.csv"))
if seed_files:
    all_dfs = [pd.read_csv(f) for f in seed_files]
    combined = pd.concat(all_dfs, ignore_index=True)
    combined.to_csv(out_dir / "coefficient_sweep_all.csv", index=False)
    print(f"\nSaved combined: coefficient_sweep_all.csv ({len(combined)} rows)")

if not combined.empty:
    num_cols = [c for c in combined.columns
                if c not in ["seed", "lambda"]
                and pd.api.types.is_numeric_dtype(combined[c])]
    agg = combined.groupby("lambda")[num_cols].agg(["mean", "std"])
    agg.to_csv(out_dir / "coefficient_sweep_agg.csv")
    print("Saved aggregation: coefficient_sweep_agg.csv")

sweep_df